# Sentinel-2 burned area and fire severity with geemap

This notebook is a Python/geemap adaptation of
`GEE/Wildfire_detection.js` and the attached JavaScript configuration. It
preserves the original simple and advanced image-selection modes, spectral
indices, dNBR2 burned-area method, water/shoreline/land-cover corrections,
minimum-patch filtering, severity classes, statistics, threshold exploration,
and Google Drive export definitions.

The main interaction change is deliberate: **running the analysis does not
automatically draw result layers**. It creates an output catalogue instead.
Select one or more catalogue entries and then:

- preview only those entries in the notebook map; or
- run the dedicated ArcGIS Pro transfer cell to add only those entries to an
  existing map or 3D Scene.

The ArcGIS transfer never calls `Map.openView()`, so it does not create a
duplicate Scene tab.

### Quick workflow

1. Run the setup and import cells.
2. Review the single **User options** cell. Defaults match the attached script.
3. Authenticate Earth Engine.
4. Keep the supplied polygon, or draw a replacement polygon/rectangle on the
   geemap map.
5. Run the analysis cell.
6. Select outputs from the catalogue.
7. Preview them in geemap or run the ArcGIS Pro transfer cell.
8. Review statistics and create/start exports only when needed.
9. Optionally run the final local raster/polygon area cross-check.

> **Scientific-use note:** the thresholds follow the source script and require
> event-specific validation. The Separability Index described by Sobrino et al.
> (2024) remains out of scope, as in the JavaScript source.

> **Security:** authentication uses Earth Engine's normal OAuth flow. Never put
> API keys, OAuth tokens, service-account JSON, or private keys in this notebook.


## 1. Setup and imports

Use a cloned ArcGIS Pro environment when installing packages for ArcGIS Pro.
The install line is intentionally commented so a normal rerun does not modify
the active environment.


In [ ]:
# Uncomment only when the selected environment does not already have these packages.
# %pip install earthengine-api geemap geedim rasterio geopandas shapely pyproj ipywidgets pandas nbformat


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import html
import math
import threading

import ee
import geemap
import geemap.common as geemap_common
import geemap.geemap as geemap_widget
import ipywidgets as widgets
import pandas as pd
from geemap import core as geemap_core
from IPython import get_ipython
from IPython.display import display

# Prevent geemap from trying to mirror every layer operation through ArcPy.
# ArcGIS transfer is handled explicitly and safely later in the notebook.
geemap_common.is_arcpy = lambda: False
geemap_widget.is_arcpy = lambda: False


## 2. User options

All user-editable variables are collected in this cell. The defaults reproduce
the attached JavaScript run, including the supplied polygon and July 2026
simple-mode dates. Internal aliases from the JavaScript are derived later and
do not need to be edited separately.


In [ ]:
# ---------------------------------------------------------------------------
# Earth Engine
# ---------------------------------------------------------------------------
PROJECT_ID = "hardy-tenure-383607"

# ---------------------------------------------------------------------------
# A) SIMPLE APPROACH — default quick run
# ---------------------------------------------------------------------------
RUN_SIMPLE_MODE = True
SIMPLE_TARGET_DATE = "2026-07-29"
SIMPLE_PRE_MODE = "manual"  # "manual" | "auto_previous"
SIMPLE_MANUAL_PRE_DATE = "2026-06-24"
SIMPLE_AUTO_PRE_LOOKBACK_DAYS = 30
SIMPLE_PRE_FULL_DATE_MOSAIC = True
SIMPLE_POST_FULL_DATE_MOSAIC = True

# ---------------------------------------------------------------------------
# B) AOI — choose exactly one source
# ---------------------------------------------------------------------------
USE_DRAWN_AOI = True
USE_ADMIN_AOI = False

# Polygon supplied at the top of the attached JavaScript. If no replacement
# polygon/rectangle is drawn on the map, this polygon is used.
DEFAULT_DRAWN_AOI_COORDINATES = [
    [
        [-3.656938890015864, 40.62992637365294],
        [-4.883287766968989, 40.625757309402076],
        [-4.903887132203364, 40.1529781457142],
        [-3.448198655640864, 40.162424260627816],
        [-3.593767503297114, 40.580923432758034],
    ]
]

ADMIN_LEVEL = 2
ADMIN_COUNTRY = "Spain"
ADMIN_NAMES = ["Ourense"]
SHOW_ADMIN_LAYER = True
ADMIN_STATS = False
ADMIN_STATS_ALL = False
ADMIN_STATS_PREVIEW_ROWS = 100

# ---------------------------------------------------------------------------
# C) SENTINEL-2 INPUTS AND CLOUD FILTERS
# ---------------------------------------------------------------------------
S2_COLLECTION = "SR"  # "SR" | "L1C"
S2_SR_IC = "COPERNICUS/S2_SR_HARMONIZED"
S2_L1C_IC = "COPERNICUS/S2_HARMONIZED"
MAX_SCENE_CLOUD = 60
SINGLE_SEARCH_DAYS = 5
REQUIRE_EXACT_DATE = False
PRINT_SELECTED_SCENES = True

# ---------------------------------------------------------------------------
# D) CORE METHOD PARAMETERS
# ---------------------------------------------------------------------------
APPLY_SMOOTHING = True
THRESH_DNBR2 = 0.10

NDVI_LOW_MAX = 0.40
NDVI_FULL_MIN = 0.75

TH_BAIS2_LOW_MAX = 0.90
TH_BAIS2_MOD_MAX = 1.00

TH_NBR_LOW_MIN = 0.00
TH_NBR_MOD_MIN = -0.30

TH_NBR3_LOW_MIN = 0.20
TH_NBR3_MOD_MIN = -0.30

# ---------------------------------------------------------------------------
# E) OPTIONAL CORRECTIONS AND MASKS
# ---------------------------------------------------------------------------
APPLY_WATER_VETO = True
WATER_VETO_INDEX = "mNDWI"  # "mNDWI" | "NDWI"
WATER_VETO_THRESHOLD = 0
WATER_VETO_WHEN = "pre_or_post"  # "pre_only" | "post_only" | "pre_or_post"
WATER_VETO_SOURCE = "both"  # "index" | "s2" | "both"
WATER_VETO_COMBINE = "union"  # "union" | "intersection"
SHOW_WATER_MASK = True
SHOW_WATER_COMPONENTS = False

APPLY_EDGE_PROTECT = True
EDGE_PROTECT_PIXELS = 1

USE_LC_MASK = False
USE_WORLDCOVER = True
WORLDCOVER_ASSET = "ESA/WorldCover/v200/2021"
YOUR_OWN_LC_ASSET = "YOUR_OWN_ASSET"

APPLY_MIN_PATCH = True
PATCH_FILTER_MODE = "pixel"  # "pixel" | "area"
MIN_PATCH_PIXELS = 25
MIN_PATCH_HA = 1
CONNECTIVITY_EIGHT = True
SHOW_PATCH_DEBUG = True

# ---------------------------------------------------------------------------
# F) OUTPUTS, VISUALIZATION, STATISTICS, AND DIAGNOSTICS
# ---------------------------------------------------------------------------
AREA_METHOD = "utm"  # "utm" | "native"
ANALYSIS_SCALE = 10
EXPORT_SCALE = 10
EXPORT_CRS = "EPSG:4326"
EXPORT_FOLDER = "GEE_Fire_Sobrino2024"
PRINT_STATS = True
EXPORT_STATS = True
TILE_SCALE = 2  # Set to None to disable.
PALETTE_SEVERITY = ["00FF00", "FFF59D", "FFA726", "EF5350"]

RUN_MULTI_THRESH = False
MULTI_THRESH = [0.06, 0.08, 0.10, 0.12, 0.14]
MULTI_ADD_TO_MAP = True
MULTI_SHOW_PREPATCH = False
MULTI_EXPORT = False

# ---------------------------------------------------------------------------
# G) ADVANCED / LEGACY DATE AND COMPOSITE SETTINGS
# Used only when RUN_SIMPLE_MODE = False.
# ---------------------------------------------------------------------------
FIRE_DATE = "2026-07-05"
PRE_DAYS = 10
POST_DAYS = 30
COMPOSITE_MODE = "single"  # "single" | "median"
PRE_SINGLE_DATE = "2026-06-25"
POST_SINGLE_DATE = "2026-07-05"

# ---------------------------------------------------------------------------
# Notebook-only output selection and ArcGIS transfer behavior
# ---------------------------------------------------------------------------
DEFAULT_SELECTED_OUTPUTS = ("burned_final", "severity")
REPLACE_NOTEBOOK_PREVIEWS = True
ZOOM_TO_SELECTED_OUTPUTS = True

# None = active map/Scene when available, otherwise the first Scene, then map.
ARCGIS_TARGET_NAME = None
ARCGIS_REPLACE_SAME_NAME = True
ARCGIS_SET_DEFAULT_CAMERA = True
ARCGIS_ADJUST_ACTIVE_CAMERA = False

# Python creates export definitions without submitting them. Set True only
# when you explicitly want the export cell to start every defined task.
AUTO_START_EXPORT_TASKS = False


# ---------------------------------------------------------------------------
# Local burned-area validation
# ---------------------------------------------------------------------------
# Relative paths are resolved from the notebook kernel's working directory.
LOCAL_VALIDATION_DIRECTORY = str(
    Path.home() / "Desktop" / "wildfire_local_validation"
)
LOCAL_VALIDATION_SCALE = ANALYSIS_SCALE
# None selects the AOI-derived UTM CRS used by the default Earth Engine stats.
LOCAL_VALIDATION_CRS = None
LOCAL_VALIDATION_CONNECTIVITY = 8  # 4 | 8 for raster polygonization.
LOCAL_VALIDATION_NUM_THREADS = 4
LOCAL_VALIDATION_OVERWRITE = True
LOCAL_VALIDATION_TOLERANCE_PERCENT = 0.5


## 3. Authenticate and capture the optional ArcGIS Pro context

Earth Engine credentials are stored outside the repository. When this notebook
runs inside ArcGIS Pro, the current project is captured on the notebook
execution thread; external kernels can still use every Earth Engine/geemap
feature but cannot modify an already-open ArcGIS Scene.


In [ ]:
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    print("Earth Engine credentials were not available. Starting OAuth authentication...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

NOTEBOOK_THREAD_ID = threading.get_ident()
try:
    NOTEBOOK_IO_LOOP = get_ipython().kernel.io_loop
except Exception:
    NOTEBOOK_IO_LOOP = None

ARCGIS_CONTEXT = {"arcpy": None, "project": None, "error": None}


def capture_current_arcgis_project():
    if threading.get_ident() != NOTEBOOK_THREAD_ID:
        raise RuntimeError(
            "Capture the ArcGIS Pro project from an ordinary notebook cell."
        )
    try:
        import arcpy
    except ImportError as error:
        ARCGIS_CONTEXT.update(
            arcpy=None,
            project=None,
            error=f"ArcPy is unavailable: {error}",
        )
        return

    try:
        project = arcpy.mp.ArcGISProject("CURRENT")
    except Exception as error:
        ARCGIS_CONTEXT.update(
            arcpy=arcpy,
            project=None,
            error=f"{type(error).__name__}: {error}",
        )
        return

    ARCGIS_CONTEXT.update(arcpy=arcpy, project=project, error=None)


capture_current_arcgis_project()
print(f"Earth Engine initialized with project: {PROJECT_ID}")
if ARCGIS_CONTEXT["project"] is not None:
    print("ArcGIS Pro bridge captured on the notebook execution thread.")
else:
    print("ArcGIS Pro bridge is optional and was not captured in this kernel.")


## 4. Choose the area of interest

The red outline is the polygon supplied with the JavaScript. To replace it,
draw one polygon or rectangle. The most recently drawn geometry is used when
`USE_DRAWN_AOI = True`. Use the clear button to return to the supplied polygon.


In [ ]:
DEFAULT_DRAWN_AOI = ee.Geometry.Polygon(DEFAULT_DRAWN_AOI_COORDINATES)
MAP = geemap.Map(center=(40.39, -4.17), zoom=8, height="620px")
try:
    MAP.add_basemap("HYBRID")
except Exception:
    pass
try:
    MAP.add("layer_manager")
except Exception:
    pass

default_aoi_outline = ee.FeatureCollection(
    [ee.Feature(DEFAULT_DRAWN_AOI)]
).style(color="FF3333", fillColor="00000000", width=3)
geemap_core.Map.add_layer(
    MAP,
    default_aoi_outline,
    {},
    "Supplied default AOI",
    True,
)

aoi_status = widgets.HTML(
    "<b>AOI:</b> The supplied polygon will be used unless you draw a replacement."
)
clear_drawn_aoi_button = widgets.Button(
    description="Use supplied polygon", icon="undo"
)


def clear_drawn_aoi(_=None):
    try:
        MAP.remove_drawn_features()
    except Exception:
        draw_control = getattr(MAP, "draw_control", None)
        if draw_control is not None:
            for method_name in ("reset", "clear"):
                method = getattr(draw_control, method_name, None)
                if callable(method):
                    method()
                    break
    try:
        MAP.user_roi = None
    except Exception:
        pass
    aoi_status.value = (
        "<b>AOI:</b> The supplied polygon will be used unless you draw a replacement."
    )


def selected_drawn_aoi():
    user_roi = getattr(MAP, "user_roi", None)
    if user_roi is None:
        return DEFAULT_DRAWN_AOI, "supplied polygon"
    if isinstance(user_roi, ee.FeatureCollection):
        return user_roi.geometry(maxError=1), "drawn feature collection"
    if isinstance(user_roi, ee.Feature):
        return user_roi.geometry(maxError=1), "drawn feature"
    return ee.Geometry(user_roi), "drawn polygon"


def note_created_geometry(_draw_control, geometry=None):
    aoi_status.value = (
        "<b>AOI:</b> A drawn geometry is ready and will replace the supplied polygon."
    )


def note_deleted_geometry(draw_control, geometry=None):
    if draw_control.count:
        aoi_status.value = (
            "<b>AOI:</b> The most recently drawn geometry will be used."
        )
    else:
        aoi_status.value = (
            "<b>AOI:</b> No drawn replacement remains; the supplied polygon will be used."
        )


try:
    MAP.draw_control.on_geometry_create(note_created_geometry)
    MAP.draw_control.on_geometry_edit(note_created_geometry)
    MAP.draw_control.on_geometry_delete(note_deleted_geometry)
except Exception:
    pass

clear_drawn_aoi_button.on_click(clear_drawn_aoi)
display(widgets.HBox([clear_drawn_aoi_button]), aoi_status)
display(MAP)


## 5. Sentinel-2 selection and preprocessing engine

These functions translate the source collection filters, same-pass and
same-date mosaics, nearest-date behavior, cloud masks, and median composites.
Earth Engine date-range end values are exclusive, as in the source script.


In [ ]:
def validate_options():
    allowed = {
        "SIMPLE_PRE_MODE": (SIMPLE_PRE_MODE, {"manual", "auto_previous"}),
        "S2_COLLECTION": (S2_COLLECTION, {"SR", "L1C"}),
        "WATER_VETO_INDEX": (WATER_VETO_INDEX, {"mNDWI", "NDWI"}),
        "WATER_VETO_WHEN": (
            WATER_VETO_WHEN,
            {"pre_only", "post_only", "pre_or_post"},
        ),
        "WATER_VETO_SOURCE": (
            WATER_VETO_SOURCE,
            {"index", "s2", "both"},
        ),
        "WATER_VETO_COMBINE": (
            WATER_VETO_COMBINE,
            {"union", "intersection"},
        ),
        "PATCH_FILTER_MODE": (PATCH_FILTER_MODE, {"pixel", "area"}),
        "AREA_METHOD": (AREA_METHOD, {"utm", "native"}),
        "COMPOSITE_MODE": (COMPOSITE_MODE, {"single", "median"}),
    }
    invalid = [
        f"{name}={value!r}; expected one of {sorted(choices)}"
        for name, (value, choices) in allowed.items()
        if value not in choices
    ]
    if USE_DRAWN_AOI == USE_ADMIN_AOI:
        invalid.append(
            "Choose exactly one AOI source: USE_DRAWN_AOI or USE_ADMIN_AOI."
        )
    if NDVI_LOW_MAX >= NDVI_FULL_MIN:
        invalid.append("NDVI_LOW_MAX must be lower than NDVI_FULL_MIN.")
    if MIN_PATCH_PIXELS < 0 or MIN_PATCH_HA < 0:
        invalid.append("Minimum patch thresholds cannot be negative.")
    if not 0 <= MAX_SCENE_CLOUD <= 100:
        invalid.append("MAX_SCENE_CLOUD must be between 0 and 100.")
    if invalid:
        raise ValueError("\n".join(invalid))


def scale_s2(image):
    reflectance = image.select("B.*").divide(10000)
    return image.addBands(reflectance, None, True)


def mask_s2_scl(image):
    scl = image.select("SCL")
    good = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )
    return image.updateMask(good)


def mask_s2_qa60(image):
    qa = image.select("QA60")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = (
        qa.bitwiseAnd(cloud_bit_mask)
        .eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )
    return image.updateMask(mask)


def s2_collection_id():
    return S2_SR_IC if S2_COLLECTION == "SR" else S2_L1C_IC


def prepared_s2_collection(aoi, start_date, end_date):
    mask_function = mask_s2_scl if S2_COLLECTION == "SR" else mask_s2_qa60
    return (
        ee.ImageCollection(s2_collection_id())
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD))
        .map(scale_s2)
        .map(mask_function)
    )


def collection_count(collection, label):
    count = int(collection.size().getInfo())
    if PRINT_SELECTED_SCENES:
        print(f"{label}: {count} matching granule(s).")
    return count


def get_s2_composite(aoi, start_date, end_date, label="median composite"):
    collection = prepared_s2_collection(aoi, start_date, end_date)
    count = collection_count(collection, label)
    if count < 1:
        raise RuntimeError(
            f"No Sentinel-2 images matched {label} after the AOI, date, "
            "and cloud filters."
        )
    return (
        collection.median()
        .clip(aoi)
        .set(
            {
                "is_composite": True,
                "composite_start": ee.Date(start_date).millis(),
                "composite_end": ee.Date(end_date).millis(),
                "mosaic_count": count,
                "selection_label": label,
            }
        )
    )


def same_pass_collection(base, first):
    datatake = ee.String(first.get("DATATAKE_IDENTIFIER"))
    by_datatake = base.filter(
        ee.Filter.eq("DATATAKE_IDENTIFIER", datatake)
    )
    time = first.date()
    by_time_orbit = (
        base.filterDate(time.advance(-2, "hour"), time.advance(2, "hour"))
        .filter(
            ee.Filter.eq(
                "SENSING_ORBIT_NUMBER",
                first.get("SENSING_ORBIT_NUMBER"),
            )
        )
        .filter(
            ee.Filter.eq(
                "SENSING_ORBIT_DIRECTION",
                first.get("SENSING_ORBIT_DIRECTION"),
            )
        )
    )
    return ee.ImageCollection(
        ee.Algorithms.If(by_datatake.size().gt(0), by_datatake, by_time_orbit)
    )


def ranked_s2_collection(aoi, target_date):
    target = ee.Date(target_date)
    start = (
        target
        if REQUIRE_EXACT_DATE
        else target.advance(-SINGLE_SEARCH_DAYS, "day")
    )
    end = (
        target.advance(1, "day")
        if REQUIRE_EXACT_DATE
        else target.advance(SINGLE_SEARCH_DAYS + 1, "day")
    )
    base = prepared_s2_collection(aoi, start, end)

    def add_rank(image):
        difference_days = ee.Number(
            image.date().difference(target, "day")
        ).abs()
        cloud = ee.Number(image.get("CLOUDY_PIXEL_PERCENTAGE"))
        return image.set("rank", difference_days.multiply(10000).add(cloud))

    return base, base.map(add_rank).sort("rank"), target


def fallback_near_date_composite(aoi, target, label):
    wide_start = target.advance(-SINGLE_SEARCH_DAYS, "day")
    wide_end = target.advance(SINGLE_SEARCH_DAYS + 1, "day")
    if PRINT_SELECTED_SCENES:
        print(
            f"{label}: no image in the requested window; using the "
            f"±{SINGLE_SEARCH_DAYS}-day median fallback."
        )
    return get_s2_composite(
        aoi,
        wide_start,
        wide_end,
        label=f"{label} fallback median",
    )


def get_s2_single_scene(aoi, target_date, label):
    base, ranked, target = ranked_s2_collection(aoi, target_date)
    if collection_count(ranked, label) < 1:
        return fallback_near_date_composite(aoi, target, label)

    first = ee.Image(ranked.first())
    same_pass = same_pass_collection(base, first)
    timestamp = ee.Number(first.get("system:time_start"))
    mosaic = (
        same_pass.mosaic()
        .clip(aoi)
        .set(
            {
                "pass_time": timestamp,
                "system:time_start": timestamp,
                "mosaic_tile_ids": same_pass.aggregate_array("system:index"),
                "mosaic_mgrs_tiles": same_pass.aggregate_array("MGRS_TILE"),
                "mosaic_count": same_pass.size(),
                "chosen_index": first.get("system:index"),
                "chosen_cloud": first.get("CLOUDY_PIXEL_PERCENTAGE"),
                "is_composite": False,
                "full_date_mosaic": False,
                "DATATAKE_IDENTIFIER": first.get("DATATAKE_IDENTIFIER"),
                "SENSING_ORBIT_NUMBER": first.get("SENSING_ORBIT_NUMBER"),
                "SENSING_ORBIT_DIRECTION": first.get(
                    "SENSING_ORBIT_DIRECTION"
                ),
                "selection_label": label,
            }
        )
    )
    return mosaic


def get_s2_single_date_all_intersecting(aoi, target_date, label):
    base, ranked, target = ranked_s2_collection(aoi, target_date)
    if collection_count(ranked, label) < 1:
        return fallback_near_date_composite(aoi, target, label)

    first = ee.Image(ranked.first())
    selected = first.date()
    day_start = ee.Date.fromYMD(
        selected.get("year"),
        selected.get("month"),
        selected.get("day"),
    )
    day_end = day_start.advance(1, "day")
    # Cloudier images first; clearer images are later and win overlaps.
    same_date = base.filterDate(day_start, day_end).sort(
        "CLOUDY_PIXEL_PERCENTAGE", False
    )
    timestamp = ee.Number(first.get("system:time_start"))
    return (
        same_date.mosaic()
        .clip(aoi)
        .set(
            {
                "pass_time": timestamp,
                "system:time_start": timestamp,
                "mosaic_tile_ids": same_date.aggregate_array("system:index"),
                "mosaic_mgrs_tiles": same_date.aggregate_array("MGRS_TILE"),
                "mosaic_datatakes": same_date.aggregate_array(
                    "DATATAKE_IDENTIFIER"
                ),
                "mosaic_count": same_date.size(),
                "chosen_index": first.get("system:index"),
                "chosen_cloud": first.get("CLOUDY_PIXEL_PERCENTAGE"),
                "is_composite": False,
                "full_date_mosaic": True,
                "selected_date": day_start.format("YYYY-MM-dd"),
                "selection_label": label,
            }
        )
    )


def get_s2_previous_pass_before(aoi, reference_image, target_date):
    reference_millis = ee.Algorithms.If(
        reference_image.get("pass_time"),
        reference_image.get("pass_time"),
        ee.Date(target_date).millis(),
    )
    reference_date = ee.Date(reference_millis)
    start = reference_date.advance(
        -SIMPLE_AUTO_PRE_LOOKBACK_DAYS, "day"
    )
    base = prepared_s2_collection(aoi, start, reference_date)

    def add_rank(image):
        difference_days = ee.Number(
            reference_date.difference(image.date(), "day")
        ).abs()
        cloud = ee.Number(image.get("CLOUDY_PIXEL_PERCENTAGE"))
        return image.set("rank", difference_days.multiply(10000).add(cloud))

    ranked = base.map(add_rank).sort("rank")
    if collection_count(ranked, "automatic previous-pass search") < 1:
        return get_s2_composite(
            aoi,
            start,
            reference_date,
            label="automatic previous-pass fallback median",
        )

    first = ee.Image(ranked.first())
    same_pass = same_pass_collection(base, first)
    timestamp = ee.Number(first.get("system:time_start"))
    return (
        same_pass.mosaic()
        .clip(aoi)
        .set(
            {
                "pass_time": timestamp,
                "system:time_start": timestamp,
                "mosaic_tile_ids": same_pass.aggregate_array("system:index"),
                "mosaic_mgrs_tiles": same_pass.aggregate_array("MGRS_TILE"),
                "mosaic_count": same_pass.size(),
                "chosen_index": first.get("system:index"),
                "chosen_cloud": first.get("CLOUDY_PIXEL_PERCENTAGE"),
                "is_composite": False,
                "quick_previous": True,
                "reference_post_time": reference_date.millis(),
                "DATATAKE_IDENTIFIER": first.get("DATATAKE_IDENTIFIER"),
                "SENSING_ORBIT_NUMBER": first.get("SENSING_ORBIT_NUMBER"),
                "SENSING_ORBIT_DIRECTION": first.get(
                    "SENSING_ORBIT_DIRECTION"
                ),
                "selection_label": "automatic previous pass",
            }
        )
    )


## 6. Indices, masks, severity, and analysis graph

The patch-mode value is spelled `"pixel"` in the source settings and comments.
This adaptation consistently uses that spelling when applying
`MIN_PATCH_PIXELS`.


In [ ]:
def add_indices(image):
    band = {
        name: image.select(name)
        for name in (
            "B2",
            "B3",
            "B4",
            "B5",
            "B6",
            "B7",
            "B8",
            "B8A",
            "B11",
            "B12",
        )
    }
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    nbr = image.normalizedDifference(["B8", "B12"]).rename("NBR")
    nbr2 = (
        band["B11"]
        .subtract(band["B12"])
        .divide(band["B11"].add(band["B12"]))
        .rename("NBR2")
    )
    nbr3 = (
        band["B7"]
        .subtract(band["B12"])
        .divide(band["B7"].add(band["B12"]))
        .rename("NBR3")
    )
    term1 = ee.Image(1).subtract(
        band["B6"]
        .multiply(band["B7"])
        .multiply(band["B8A"])
        .divide(band["B4"])
        .sqrt()
    )
    difference = band["B12"].subtract(band["B8A"])
    term2 = difference.divide(difference.abs().sqrt().add(1))
    bais2 = term1.multiply(term2).rename("BAIS2")
    bai = (
        ee.Image(1)
        .divide(
            ee.Image(0.1)
            .subtract(band["B4"])
            .pow(2)
            .add(ee.Image(0.06).subtract(band["B8"]).pow(2))
        )
        .rename("BAI")
    )
    mirbi = (
        band["B11"].multiply(10)
        .subtract(band["B12"].multiply(9.8))
        .rename("MIRBI")
    )
    return image.addBands([ndvi, nbr, nbr2, nbr3, bais2, bai, mirbi])


def as_mask(image):
    return ee.Image(0) if image is None else ee.Image(image).gt(0)


def mask_or(first, second):
    return as_mask(first).Or(as_mask(second))


def mask_and(first, second):
    return as_mask(first).And(as_mask(second))


def calculate_ndwi(image):
    return image.normalizedDifference(["B3", "B8"]).rename("NDWI")


def calculate_mndwi(image):
    return image.normalizedDifference(["B3", "B11"]).rename("mNDWI")


def water_mask_from_index(image):
    index = (
        calculate_mndwi(image)
        if WATER_VETO_INDEX == "mNDWI"
        else calculate_ndwi(image)
    )
    return index.gt(ee.Number(WATER_VETO_THRESHOLD)).rename("WATER_IDX")


def water_mask_from_scl(image):
    return image.select("SCL").eq(6).rename("WATER_SCL")


def combine_pre_post_masks(pre_mask, post_mask):
    if WATER_VETO_WHEN == "pre_only":
        return as_mask(pre_mask)
    if WATER_VETO_WHEN == "post_only":
        return as_mask(post_mask)
    return mask_or(pre_mask, post_mask)


def woody_shrub_mask(aoi):
    if USE_WORLDCOVER:
        worldcover = ee.Image(WORLDCOVER_ASSET).select("Map")
        return (
            worldcover.eq(10)
            .Or(worldcover.eq(20))
            .selfMask()
            .clip(aoi)
        )
    land_cover = ee.Image(YOUR_OWN_LC_ASSET).select(0).clip(aoi)
    woody = land_cover.eq(111).Or(land_cover.eq(112))
    shrub = (
        land_cover.eq(120)
        .Or(land_cover.eq(121))
        .Or(land_cover.eq(122))
    )
    return woody.Or(shrub).selfMask()


def classify_low_density(post):
    bais2 = post.select("BAIS2")
    low = bais2.lt(TH_BAIS2_LOW_MAX)
    moderate = bais2.gte(TH_BAIS2_LOW_MAX).And(
        bais2.lt(TH_BAIS2_MOD_MAX)
    )
    high = bais2.gte(TH_BAIS2_MOD_MAX)
    return (
        ee.Image(0)
        .where(low, 1)
        .where(moderate, 2)
        .where(high, 3)
        .rename("SEV_LOW")
    )


def classify_mixed_density(post):
    nbr = post.select("NBR")
    low = nbr.gt(TH_NBR_LOW_MIN)
    moderate = nbr.lte(TH_NBR_LOW_MIN).And(nbr.gt(TH_NBR_MOD_MIN))
    high = nbr.lte(TH_NBR_MOD_MIN)
    return (
        ee.Image(0)
        .where(low, 1)
        .where(moderate, 2)
        .where(high, 3)
        .rename("SEV_MIX")
    )


def classify_full_density(post):
    nbr3 = post.select("NBR3")
    low = nbr3.gt(TH_NBR3_LOW_MIN)
    moderate = nbr3.lte(TH_NBR3_LOW_MIN).And(
        nbr3.gt(TH_NBR3_MOD_MIN)
    )
    high = nbr3.lte(TH_NBR3_MOD_MIN)
    return (
        ee.Image(0)
        .where(low, 1)
        .where(moderate, 2)
        .where(high, 3)
        .rename("SEV_FULL")
    )


def minimum_patch_pixels(mask_image):
    if not APPLY_MIN_PATCH:
        return ee.Number(0)
    if PATCH_FILTER_MODE == "pixel":
        return ee.Number(MIN_PATCH_PIXELS).max(0).ceil()
    nominal_scale = mask_image.projection().nominalScale()
    return (
        ee.Number(MIN_PATCH_HA)
        .multiply(10000)
        .divide(nominal_scale.multiply(nominal_scale))
        .ceil()
    )


def apply_minimum_patch(mask_image, label):
    if not APPLY_MIN_PATCH:
        return mask_image
    minimum_pixels = minimum_patch_pixels(mask_image)
    safe_maximum = ee.Number(1024).min(minimum_pixels).max(2)
    connected = mask_image.connectedPixelCount(
        maxSize=safe_maximum,
        eightConnected=CONNECTIVITY_EIGHT,
    )
    if SHOW_PATCH_DEBUG:
        print(
            f"Patch filter [{label}]: mode={PATCH_FILTER_MODE}, "
            f"minimum={MIN_PATCH_PIXELS if PATCH_FILTER_MODE == 'pixel' else MIN_PATCH_HA}, "
            f"eight_connected={CONNECTIVITY_EIGHT}"
        )
    return mask_image.updateMask(connected.gte(minimum_pixels))


def admin_collection():
    collection_id = {
        0: "FAO/GAUL/2015/level0",
        1: "FAO/GAUL/2015/level1",
        2: "FAO/GAUL/2015/level2",
    }.get(ADMIN_LEVEL)
    if collection_id is None:
        raise ValueError("ADMIN_LEVEL must be 0, 1, or 2.")
    name_field = {
        0: "ADM0_NAME",
        1: "ADM1_NAME",
        2: "ADM2_NAME",
    }[ADMIN_LEVEL]
    collection = ee.FeatureCollection(collection_id).filter(
        ee.Filter.eq("ADM0_NAME", ADMIN_COUNTRY)
    )
    if not ADMIN_STATS_ALL:
        collection = collection.filter(
            ee.Filter.inList(name_field, ADMIN_NAMES)
        )
    return collection, name_field


def analysis_aoi():
    if USE_ADMIN_AOI:
        collection, _name_field = admin_collection()
        if int(collection.size().getInfo()) < 1:
            raise RuntimeError(
                "The GAUL settings returned no administrative features."
            )
        return collection.geometry(maxError=1), collection, "GAUL admin AOI"
    geometry, source_label = selected_drawn_aoi()
    return geometry, None, source_label


def utm_crs_for_geometry(geometry):
    centroid = ee.Geometry(geometry).centroid(maxError=1)
    longitude = ee.Number(centroid.coordinates().get(0))
    latitude = ee.Number(centroid.coordinates().get(1))
    zone = longitude.add(180).divide(6).floor().add(1)
    root = ee.String(
        ee.Algorithms.If(latitude.gte(0), "EPSG:326", "EPSG:327")
    )
    return root.cat(zone.format("%02d"))


def selected_image_summary(image, role):
    property_names = [
        "selection_label",
        "selected_date",
        "pass_time",
        "mosaic_count",
        "mosaic_mgrs_tiles",
        "mosaic_datatakes",
        "DATATAKE_IDENTIFIER",
        "SENSING_ORBIT_NUMBER",
        "SENSING_ORBIT_DIRECTION",
        "chosen_index",
        "chosen_cloud",
        "is_composite",
        "full_date_mosaic",
        "composite_start",
        "composite_end",
    ]
    values = image.toDictionary(property_names).getInfo()
    pass_time = values.get("pass_time")
    selected_time = (
        datetime.fromtimestamp(pass_time / 1000, timezone.utc).isoformat()
        if pass_time
        else None
    )
    return {
        "role": role,
        "selection": values.get("selection_label"),
        "selected_date": values.get("selected_date"),
        "selected_time_utc": selected_time,
        "granule_count": values.get("mosaic_count"),
        "MGRS_tiles": values.get("mosaic_mgrs_tiles"),
        "datatakes": values.get("mosaic_datatakes"),
        "chosen_index": values.get("chosen_index"),
        "chosen_cloud_percent": values.get("chosen_cloud"),
        "is_composite": values.get("is_composite"),
        "full_date_mosaic": values.get("full_date_mosaic"),
    }


def run_analysis():
    validate_options()
    aoi, admin_fc, aoi_source = analysis_aoi()
    fire_date = SIMPLE_TARGET_DATE if RUN_SIMPLE_MODE else FIRE_DATE

    if RUN_SIMPLE_MODE:
        post_raw = (
            get_s2_single_date_all_intersecting(
                aoi,
                SIMPLE_TARGET_DATE,
                "simple post/target full-date mosaic",
            )
            if SIMPLE_POST_FULL_DATE_MOSAIC
            else get_s2_single_scene(
                aoi,
                SIMPLE_TARGET_DATE,
                "simple post/target pass mosaic",
            )
        )
        if SIMPLE_PRE_MODE == "manual":
            pre_raw = (
                get_s2_single_date_all_intersecting(
                    aoi,
                    SIMPLE_MANUAL_PRE_DATE,
                    "simple manual pre full-date mosaic",
                )
                if SIMPLE_PRE_FULL_DATE_MOSAIC
                else get_s2_single_scene(
                    aoi,
                    SIMPLE_MANUAL_PRE_DATE,
                    "simple manual pre pass mosaic",
                )
            )
        else:
            pre_raw = get_s2_previous_pass_before(
                aoi,
                post_raw,
                SIMPLE_TARGET_DATE,
            )
    else:
        fire = ee.Date(FIRE_DATE)
        pre_start = fire.advance(-PRE_DAYS, "day")
        pre_end = fire.advance(-1, "day")
        post_start = fire
        post_end = fire.advance(POST_DAYS, "day")
        if COMPOSITE_MODE == "single":
            pre_raw = get_s2_single_scene(
                aoi,
                PRE_SINGLE_DATE,
                "advanced pre pass mosaic",
            )
            post_raw = get_s2_single_scene(
                aoi,
                POST_SINGLE_DATE,
                "advanced post pass mosaic",
            )
        else:
            pre_raw = get_s2_composite(
                aoi,
                pre_start,
                pre_end,
                label="advanced pre median composite",
            )
            post_raw = get_s2_composite(
                aoi,
                post_start,
                post_end,
                label="advanced post median composite",
            )

    required_bands = {
        "B2",
        "B3",
        "B4",
        "B5",
        "B6",
        "B7",
        "B8",
        "B8A",
        "B11",
        "B12",
    }
    for role, image in (("pre", pre_raw), ("post", post_raw)):
        bands = set(image.bandNames().getInfo())
        missing = sorted(required_bands - bands)
        if missing:
            raise RuntimeError(
                f"The selected {role}-fire image is missing bands: {missing}."
            )

    pre = add_indices(pre_raw)
    post = add_indices(post_raw)
    dnbr2 = pre.select("NBR2").subtract(
        post.select("NBR2")
    ).rename("dNBR2")

    index_pre = (
        water_mask_from_index(pre) if APPLY_WATER_VETO else None
    )
    index_post = (
        water_mask_from_index(post) if APPLY_WATER_VETO else None
    )
    has_scl = S2_COLLECTION == "SR"
    scl_pre = (
        water_mask_from_scl(pre)
        if APPLY_WATER_VETO and has_scl
        else None
    )
    scl_post = (
        water_mask_from_scl(post)
        if APPLY_WATER_VETO and has_scl
        else None
    )
    if (
        APPLY_WATER_VETO
        and not has_scl
        and WATER_VETO_SOURCE != "index"
    ):
        print(
            "SCL water is unavailable for L1C; using the index water mask."
        )

    water_index = (
        combine_pre_post_masks(index_pre, index_post)
        if APPLY_WATER_VETO
        else None
    )
    water_scl = (
        combine_pre_post_masks(scl_pre, scl_post)
        if APPLY_WATER_VETO and has_scl
        else None
    )
    water_mask = None
    if APPLY_WATER_VETO:
        if WATER_VETO_SOURCE == "index":
            water_mask = water_index
        elif WATER_VETO_SOURCE == "s2":
            water_mask = water_scl if has_scl else water_index
        elif has_scl:
            water_mask = (
                mask_and(water_scl, water_index)
                if WATER_VETO_COMBINE == "intersection"
                else mask_or(water_scl, water_index)
            )
        else:
            water_mask = water_index

    edge_ring = None
    if APPLY_EDGE_PROTECT:
        base_water = as_mask(water_mask)
        if not APPLY_WATER_VETO:
            base_water = mask_or(
                water_mask_from_index(pre),
                water_mask_from_index(post),
            )
        edge_ring = (
            base_water.focalMax(
                radius=EDGE_PROTECT_PIXELS,
                units="pixels",
            )
            .And(base_water.Not())
            .rename("WATER_EDGE")
        )

    water_for_visualization = as_mask(water_mask)
    dnbr2_for_smoothing = dnbr2
    if APPLY_WATER_VETO:
        dnbr2_for_smoothing = dnbr2_for_smoothing.updateMask(
            water_for_visualization.Not()
        )
    if edge_ring is not None:
        dnbr2_for_smoothing = dnbr2_for_smoothing.updateMask(
            edge_ring.Not()
        )
    dnbr2_smoothed = (
        dnbr2_for_smoothing.focalMean(radius=1, units="pixels")
        if APPLY_SMOOTHING
        else dnbr2_for_smoothing
    )

    burned = dnbr2_smoothed.gte(THRESH_DNBR2).rename("BURNED")
    if APPLY_WATER_VETO:
        burned = burned.updateMask(water_for_visualization.Not())
    if edge_ring is not None:
        burned = burned.updateMask(edge_ring.Not())

    woody_mask = None
    if USE_LC_MASK:
        woody_mask = woody_shrub_mask(aoi)
        burned = burned.updateMask(woody_mask)

    burned_pre_patch = burned
    burned = apply_minimum_patch(burned, "burned")

    multi_threshold_outputs = {}
    if RUN_MULTI_THRESH:
        for threshold in MULTI_THRESH:
            threshold_label = f"{threshold:.2f}"
            threshold_safe = f"{round(threshold * 100):02d}"
            raw_mask = dnbr2_smoothed.gte(threshold).rename(
                f"BURNED_thr_{threshold_safe}"
            )
            if APPLY_WATER_VETO:
                raw_mask = raw_mask.updateMask(
                    water_for_visualization.Not()
                )
            if edge_ring is not None:
                raw_mask = raw_mask.updateMask(edge_ring.Not())
            if woody_mask is not None:
                raw_mask = raw_mask.updateMask(woody_mask)
            final_mask = apply_minimum_patch(
                raw_mask, f"multi_{threshold_safe}"
            )
            multi_threshold_outputs[threshold_label] = {
                "pre_patch": raw_mask.selfMask(),
                "final": final_mask.selfMask(),
            }

    ndvi_pre = pre.select("NDVI")
    low_density = ndvi_pre.lt(NDVI_LOW_MAX)
    full_density = ndvi_pre.gte(NDVI_FULL_MIN)
    mixed_density = ndvi_pre.gte(NDVI_LOW_MAX).And(
        ndvi_pre.lt(NDVI_FULL_MIN)
    )
    severity_low = classify_low_density(post).updateMask(low_density)
    severity_mixed = classify_mixed_density(post).updateMask(
        mixed_density
    )
    severity_full = classify_full_density(post).updateMask(full_density)
    severity_inside = (
        ee.Image(0)
        .where(severity_low.gt(0), severity_low)
        .where(severity_mixed.gt(0), severity_mixed)
        .where(severity_full.gt(0), severity_full)
        .rename("SEVERITY")
    )
    severity = (
        ee.Image(0)
        .where(burned.eq(1), severity_inside)
        .rename("SEVERITY")
    )

    selection_table = pd.DataFrame(
        [
            selected_image_summary(pre_raw, "pre-fire"),
            selected_image_summary(post_raw, "post-fire"),
        ]
    )

    return {
        "aoi": aoi,
        "aoi_source": aoi_source,
        "admin_fc": admin_fc,
        "fire_date": fire_date,
        "pre_raw": pre_raw,
        "post_raw": post_raw,
        "pre": pre,
        "post": post,
        "dnbr2": dnbr2,
        "dnbr2_smoothed": dnbr2_smoothed,
        "water_index": water_index,
        "water_scl": water_scl,
        "water_mask": water_for_visualization,
        "edge_ring": edge_ring,
        "woody_mask": woody_mask,
        "burned_pre_patch": burned_pre_patch,
        "burned": burned,
        "severity": severity,
        "multi_threshold_outputs": multi_threshold_outputs,
        "selection_table": selection_table,
    }


## 7. Run the analysis

This cell builds the Earth Engine analysis graph and checks that both selected
images contain the required bands. It still does **not** add result layers.


In [ ]:
ANALYSIS = run_analysis()
print(
    f"Analysis ready for {ANALYSIS['fire_date']} using "
    f"{ANALYSIS['aoi_source']}."
)
display(ANALYSIS["selection_table"])


## 8. Output catalogue and selective notebook preview

The catalogue replaces the source script's automatic `Map.addLayer(...)`
sequence. Select one or more entries (Ctrl/Cmd-click for several), then preview
them only if desired. The same selection is reused by the ArcGIS Pro transfer
cell below.


In [ ]:
FALSE_COLOR_VIS = {"bands": ["B12", "B8", "B4"], "min": 0.02, "max": 0.4}
NDVI_VIS = {
    "min": -0.2,
    "max": 0.9,
    "palette": [
        "7F0000",
        "CE7E45",
        "FCD163",
        "66A000",
        "207A00",
        "056201",
        "004C00",
        "023B01",
        "012E01",
        "011D01",
    ],
}


def output_entry(label, category, image, vis_params, description):
    return {
        "label": label,
        "category": category,
        "image": image,
        "vis_params": vis_params,
        "description": description,
    }


def build_output_catalogue(analysis):
    outputs = {
        "pre_false_color": output_entry(
            "Pre-fire Sentinel-2 (SWIR/NIR/Red)",
            "Imagery",
            analysis["pre"],
            FALSE_COLOR_VIS,
            "Selected pre-fire image/mosaic/composite.",
        ),
        "post_false_color": output_entry(
            "Post-fire Sentinel-2 (SWIR/NIR/Red)",
            "Imagery",
            analysis["post"],
            FALSE_COLOR_VIS,
            "Selected post-fire image/mosaic/composite.",
        ),
        "ndvi_pre": output_entry(
            "NDVI pre-fire",
            "Index",
            analysis["pre"].select("NDVI"),
            NDVI_VIS,
            "Pre-fire normalized difference vegetation index.",
        ),
        "dnbr2_smoothed": output_entry(
            "dNBR2 (smoothed/masked)",
            "Index",
            analysis["dnbr2_smoothed"],
            {
                "min": -0.2,
                "max": 0.6,
                "palette": ["white", "black", "purple", "red"],
            },
            "dNBR2 after optional water/edge masking and smoothing.",
        ),
        "burned_pre_patch": output_entry(
            "Burned mask (before patch filter)",
            "Burned area",
            analysis["burned_pre_patch"].selfMask(),
            {"palette": ["1f78b4"]},
            "Thresholded burned mask before the minimum-patch filter.",
        ),
        "burned_final": output_entry(
            f"Burned mask (final; dNBR2 ≥ {THRESH_DNBR2})",
            "Burned area",
            analysis["burned"].selfMask(),
            {"palette": ["d73027"]},
            "Final burned-area mask used for statistics and severity.",
        ),
        "severity": output_entry(
            "Fire severity (low/moderate/high)",
            "Severity",
            analysis["severity"].updateMask(
                analysis["severity"].gt(0)
            ),
            {"min": 0, "max": 3, "palette": PALETTE_SEVERITY},
            "Severity classes inside the final burned-area mask.",
        ),
    }

    if SHOW_WATER_MASK and APPLY_WATER_VETO:
        outputs["water_final"] = output_entry(
            "Water mask (final)",
            "Correction mask",
            analysis["water_mask"].selfMask(),
            {"palette": ["6baed6"], "opacity": 0.5},
            "Combined water veto applied to the analysis.",
        )
    if SHOW_WATER_COMPONENTS and analysis["water_index"] is not None:
        outputs["water_index"] = output_entry(
            "Water index component",
            "Correction mask",
            analysis["water_index"].selfMask(),
            {"palette": ["41b6c4"], "opacity": 0.5},
            "Pre/post NDWI or mNDWI water component.",
        )
    if SHOW_WATER_COMPONENTS and analysis["water_scl"] is not None:
        outputs["water_scl"] = output_entry(
            "Sentinel-2 SCL water component",
            "Correction mask",
            analysis["water_scl"].selfMask(),
            {"palette": ["2b8cbe"], "opacity": 0.5},
            "Pre/post Sentinel-2 SCL class-6 water component.",
        )
    if SHOW_WATER_MASK and analysis["edge_ring"] is not None:
        outputs["water_edge"] = output_entry(
            f"Shoreline edge ring ({EDGE_PROTECT_PIXELS} px)",
            "Correction mask",
            analysis["edge_ring"].selfMask(),
            {"palette": ["08519c"], "opacity": 0.5},
            "Optional water-edge exclusion ring.",
        )
    if USE_LC_MASK and analysis["woody_mask"] is not None:
        outputs["woody_shrub_mask"] = output_entry(
            "Woody/shrub land-cover mask",
            "Correction mask",
            analysis["woody_mask"],
            {"palette": ["228B22"], "opacity": 0.5},
            "Land-cover restriction applied to burned detections.",
        )
    if USE_ADMIN_AOI and SHOW_ADMIN_LAYER and analysis["admin_fc"] is not None:
        styled_admin = analysis["admin_fc"].style(
            color="222222", fillColor="00000000", width=2
        )
        outputs["admin_aoi"] = output_entry(
            f"Admin AOI (GAUL level {ADMIN_LEVEL})",
            "Reference",
            styled_admin,
            {},
            "Selected administrative boundary.",
        )
    if RUN_MULTI_THRESH and MULTI_ADD_TO_MAP:
        for threshold, threshold_outputs in analysis[
            "multi_threshold_outputs"
        ].items():
            safe = threshold.replace(".", "_")
            if MULTI_SHOW_PREPATCH:
                outputs[f"threshold_{safe}_pre"] = output_entry(
                    f"Tuning pre-patch dNBR2 ≥ {threshold}",
                    "Threshold tuning",
                    threshold_outputs["pre_patch"],
                    {"palette": ["1f78b4"]},
                    "Multi-threshold diagnostic before patch filtering.",
                )
            outputs[f"threshold_{safe}_final"] = output_entry(
                f"Tuning dNBR2 ≥ {threshold}",
                "Threshold tuning",
                threshold_outputs["final"],
                {"palette": ["e41a1c"]},
                "Multi-threshold diagnostic after patch filtering.",
            )
    return outputs


OUTPUTS = build_output_catalogue(ANALYSIS)
catalogue_table = pd.DataFrame(
    [
        {
            "key": key,
            "category": entry["category"],
            "output": entry["label"],
            "description": entry["description"],
        }
        for key, entry in OUTPUTS.items()
    ]
)
display(catalogue_table)

output_selector = widgets.SelectMultiple(
    options=[
        (f"{entry['category']} — {entry['label']}", key)
        for key, entry in OUTPUTS.items()
    ],
    value=tuple(
        key for key in DEFAULT_SELECTED_OUTPUTS if key in OUTPUTS
    ),
    description="Outputs:",
    layout=widgets.Layout(width="760px", height="260px"),
)
preview_button = widgets.Button(
    description="Preview selected in notebook",
    icon="eye",
    button_style="success",
)
clear_preview_button = widgets.Button(
    description="Clear result previews",
    icon="trash",
)
preview_status = widgets.HTML(
    "<b>Preview:</b> No result layer has been added automatically."
)
PREVIEW_LAYER_NAMES = []


def safe_remove_ee_layer(layer_name):
    try:
        MAP.remove_ee_layer(layer_name)
    except Exception:
        pass


def center_geometry(geometry):
    coordinates = (
        ee.Geometry(geometry)
        .bounds(maxError=1)
        .getInfo()["coordinates"][0]
    )
    x_values = [coordinate[0] for coordinate in coordinates]
    y_values = [coordinate[1] for coordinate in coordinates]
    xmin, xmax = min(x_values), max(x_values)
    ymin, ymax = min(y_values), max(y_values)
    MAP.center = ((ymin + ymax) / 2, (xmin + xmax) / 2)
    span = max(xmax - xmin, (ymax - ymin) * 1.6, 1e-6)
    MAP.zoom = max(1, min(18, math.floor(math.log2(360 / span))))


def clear_output_previews(_=None):
    for layer_name in list(PREVIEW_LAYER_NAMES):
        safe_remove_ee_layer(layer_name)
    PREVIEW_LAYER_NAMES.clear()
    preview_status.value = "<b>Preview:</b> Result previews cleared."


def preview_selected_outputs(_=None):
    selected_keys = tuple(output_selector.value)
    if not selected_keys:
        preview_status.value = (
            "<b>Preview:</b> <span style='color:#b00020'>"
            "Select at least one output.</span>"
        )
        return
    preview_button.disabled = True
    try:
        if REPLACE_NOTEBOOK_PREVIEWS:
            clear_output_previews()
        for key in selected_keys:
            entry = OUTPUTS[key]
            layer_name = f"Wildfire | {entry['label']}"
            safe_remove_ee_layer(layer_name)
            geemap_core.Map.add_layer(
                MAP,
                entry["image"],
                entry["vis_params"],
                layer_name,
                True,
            )
            if layer_name not in PREVIEW_LAYER_NAMES:
                PREVIEW_LAYER_NAMES.append(layer_name)
        if ZOOM_TO_SELECTED_OUTPUTS:
            center_geometry(ANALYSIS["aoi"])
        preview_status.value = (
            f"<b>Preview:</b> Added {len(selected_keys)} selected output(s)."
        )
    except Exception as error:
        preview_status.value = (
            f"<b>Preview:</b> <span style='color:#b00020'>"
            f"{type(error).__name__}: {html.escape(str(error))}</span>"
        )
    finally:
        preview_button.disabled = False


preview_button.on_click(preview_selected_outputs)
clear_preview_button.on_click(clear_output_previews)
display(
    output_selector,
    widgets.HBox([preview_button, clear_preview_button]),
    preview_status,
)


## 9. Send selected outputs to an existing ArcGIS Pro Scene

Choose the output(s) in the catalogue above and a target below. Then run the
next transfer cell. ArcPy work is intentionally performed by an ordinary
notebook cell, not a widget callback. The transfer adds Earth Engine web-tile
layers to the existing map/Scene and never opens a new view.


In [ ]:
def arcgis_project():
    if ARCGIS_CONTEXT["project"] is None:
        capture_current_arcgis_project()
    if ARCGIS_CONTEXT["project"] is None:
        detail = ARCGIS_CONTEXT["error"] or "unknown initialization error"
        raise RuntimeError(
            "The current ArcGIS Pro project is unavailable "
            f"({detail}). Run this notebook inside ArcGIS Pro."
        )
    return ARCGIS_CONTEXT["arcpy"], ARCGIS_CONTEXT["project"]


def arcgis_map_options():
    if ARCGIS_CONTEXT["project"] is None:
        return [("ArcGIS Pro not connected", "__UNAVAILABLE__")]
    project = ARCGIS_CONTEXT["project"]
    return [
        (f"{project_map.name} ({project_map.mapType})", project_map.name)
        for project_map in project.listMaps()
    ]


def preferred_arcgis_target(options):
    values = [value for _label, value in options]
    if ARCGIS_TARGET_NAME in values:
        return ARCGIS_TARGET_NAME
    if ARCGIS_CONTEXT["project"] is None:
        return values[0]
    project = ARCGIS_CONTEXT["project"]
    active_map = project.activeMap
    if active_map is not None and active_map.name in values:
        return active_map.name
    scenes = [
        project_map.name
        for project_map in project.listMaps()
        if project_map.mapType == "SCENE"
    ]
    return scenes[0] if scenes else values[0]


arcgis_options = arcgis_map_options()
arcgis_target_selector = widgets.Dropdown(
    options=arcgis_options,
    value=preferred_arcgis_target(arcgis_options),
    description="ArcGIS target:",
    layout=widgets.Layout(width="520px"),
)
display(arcgis_target_selector)


def earth_engine_tile_url(image, vis_params):
    map_id = ee.Image(image).getMapId(vis_params)
    tile_fetcher = map_id.get("tile_fetcher")
    url = getattr(tile_fetcher, "url_format", None)
    if not url:
        raise RuntimeError("Earth Engine did not return a tile-service URL.")
    return url


def aoi_extent(geometry):
    coordinates = (
        ee.Geometry(geometry)
        .bounds(maxError=1)
        .getInfo()["coordinates"][0]
    )
    x_values = [coordinate[0] for coordinate in coordinates]
    y_values = [coordinate[1] for coordinate in coordinates]
    return min(x_values), min(y_values), max(x_values), max(y_values)


def send_selected_outputs_to_arcgis():
    if threading.get_ident() != NOTEBOOK_THREAD_ID:
        raise RuntimeError(
            "Run the ArcGIS transfer from the ordinary notebook cell below."
        )
    selected_keys = tuple(output_selector.value)
    if not selected_keys:
        raise RuntimeError("Select at least one output from the catalogue.")

    arcpy, project = arcgis_project()
    target_name = arcgis_target_selector.value
    matches = project.listMaps(target_name)
    if not matches:
        raise RuntimeError(
            f"ArcGIS target {target_name!r} is no longer available."
        )
    target_map = matches[0]
    xmin, ymin, xmax, ymax = aoi_extent(ANALYSIS["aoi"])
    extent = arcpy.Extent(
        xmin,
        ymin,
        xmax,
        ymax,
        spatial_reference=arcpy.SpatialReference(4326),
    )

    added_names = []
    for key in selected_keys:
        entry = OUTPUTS[key]
        layer_name = f"Wildfire | {entry['label']}"
        if ARCGIS_REPLACE_SAME_NAME:
            try:
                matching_layers = list(target_map.listLayers(layer_name))
            except Exception:
                matching_layers = []
            for existing_layer in matching_layers:
                try:
                    target_map.removeLayer(existing_layer)
                except Exception:
                    pass

        tile_url = earth_engine_tile_url(
            entry["image"],
            entry["vis_params"],
        )
        added_layer = target_map.addDataFromPath(tile_url)
        if added_layer is not None:
            try:
                added_layer.name = layer_name
            except (AttributeError, RuntimeError):
                pass
        added_names.append(layer_name)

    if ARCGIS_SET_DEFAULT_CAMERA:
        target_map.defaultCamera.setExtent(extent)

    if ARCGIS_ADJUST_ACTIVE_CAMERA:
        # Never call target_map.openView(): that creates a duplicate tab.
        active_map = project.activeMap
        active_view = project.activeView
        target_is_active = (
            active_map is not None
            and active_map.name == target_map.name
        )
        if (
            target_is_active
            and active_view is not None
            and hasattr(active_view, "camera")
        ):
            active_view.camera.setExtent(extent)

    return {
        "target": f"{target_map.name} ({target_map.mapType})",
        "layers": added_names,
    }


In [ ]:
# ArcGIS Pro only: rerun this cell whenever you change the output selection.
if ARCGIS_CONTEXT["project"] is None:
    print(
        "ArcGIS Pro is not connected in this kernel. The selected outputs "
        "remain available for geemap preview."
    )
else:
    transfer_result = send_selected_outputs_to_arcgis()
    print(
        f"Added {len(transfer_result['layers'])} selected output(s) to "
        f"{transfer_result['target']}:"
    )
    for transferred_layer in transfer_result["layers"]:
        print(f"- {transferred_layer}")


## 10. Optional statistics

Skip the following code cell when you do not need area statistics or statistics exports. Scene visualization is already available above and does not depend on `STATISTICS`. Statistics are evaluated only after the analysis graph exists. Pixel area is
never reprojected; a stable AOI-derived UTM CRS is passed to reducers when
`AREA_METHOD = "utm"`.


In [ ]:
def reducer_arguments(aoi, reducer, crs_override=None):
    arguments = {
        "reducer": reducer,
        "geometry": aoi,
        "scale": ANALYSIS_SCALE,
        "maxPixels": 1e13,
    }
    if AREA_METHOD == "utm":
        arguments["crs"] = crs_override or utm_crs_for_geometry(aoi)
    if TILE_SCALE is not None:
        arguments["tileScale"] = TILE_SCALE
    return arguments


def number_or_zero(value):
    return ee.Number(ee.Algorithms.If(value, value, 0))


def build_admin_statistics(analysis, area_image, utm_crs):
    if not (USE_ADMIN_AOI and ADMIN_STATS):
        return None

    name_field = {
        0: "ADM0_NAME",
        1: "ADM1_NAME",
        2: "ADM2_NAME",
    }[ADMIN_LEVEL]
    code_field = {
        0: "ADM0_CODE",
        1: "ADM1_CODE",
        2: "ADM2_CODE",
    }[ADMIN_LEVEL]
    features = analysis["admin_fc"]
    if ADMIN_STATS_ALL:
        features, _ = admin_collection()

    def reduce_feature(feature):
        geometry = feature.geometry()

        def area_hectares(mask):
            result = area_image.updateMask(mask).reduceRegion(
                **reducer_arguments(
                    geometry,
                    ee.Reducer.sum(),
                    crs_override=utm_crs,
                )
            )
            return number_or_zero(result.get("area")).divide(10000)

        return feature.set(
            {
                "version": "v1.9n",
                "fire_date": analysis["fire_date"],
                "admin_level": ADMIN_LEVEL,
                "admin_name": feature.get(name_field),
                "admin_code": feature.get(code_field),
                "ha_total": area_hectares(analysis["burned"]),
                "ha_low": area_hectares(analysis["severity"].eq(1)),
                "ha_moderate": area_hectares(
                    analysis["severity"].eq(2)
                ),
                "ha_high": area_hectares(analysis["severity"].eq(3)),
                "area_method": AREA_METHOD,
                "scale_m": ANALYSIS_SCALE,
                "utm_crs": utm_crs,
            }
        )

    return features.map(reduce_feature)


def compute_statistics(analysis):
    area_image = ee.Image.pixelArea().rename("area")
    total_result = area_image.updateMask(
        analysis["burned"]
    ).reduceRegion(
        **reducer_arguments(analysis["aoi"], ee.Reducer.sum())
    )
    total_hectares = (
        number_or_zero(total_result.get("area"))
        .divide(10000)
        .getInfo()
    )

    grouped_result = area_image.addBands(
        analysis["severity"]
    ).reduceRegion(
        **reducer_arguments(
            analysis["aoi"],
            ee.Reducer.sum().group(
                groupField=1,
                groupName="severity",
            ),
        )
    )
    groups = ee.List(
        ee.Dictionary(grouped_result).get("groups", ee.List([]))
    ).getInfo()
    group_lookup = {
        int(group["severity"]): float(group["sum"]) / 10000
        for group in groups
        if int(group["severity"]) in (1, 2, 3)
    }
    class_names = {1: "low", 2: "moderate", 3: "high"}
    rows = [
        {
            "version": "v1.9n",
            "fire_date": analysis["fire_date"],
            "severity": severity_class,
            "class": class_names[severity_class],
            "area_m2": group_lookup.get(severity_class, 0) * 10000,
            "area_ha": group_lookup.get(severity_class, 0),
        }
        for severity_class in (1, 2, 3)
    ]
    severity_table = pd.DataFrame(rows)
    severity_fc = ee.FeatureCollection(
        [ee.Feature(None, row) for row in rows]
    )

    utm_crs = (
        utm_crs_for_geometry(analysis["aoi"]).getInfo()
        if AREA_METHOD == "utm"
        else None
    )
    admin_fc = build_admin_statistics(
        analysis,
        area_image,
        utm_crs,
    )
    admin_table = None
    if admin_fc is not None:
        preview_features = (
            admin_fc.limit(ADMIN_STATS_PREVIEW_ROWS)
            .getInfo()
            .get("features", [])
        )
        admin_table = pd.DataFrame(
            feature.get("properties", {})
            for feature in preview_features
        )

    return {
        "total_burned_ha": total_hectares,
        "utm_crs": utm_crs,
        "severity_table": severity_table,
        "severity_fc": severity_fc,
        "admin_fc": admin_fc,
        "admin_table": admin_table,
    }


STATISTICS = (
    compute_statistics(ANALYSIS)
    if PRINT_STATS or EXPORT_STATS or ADMIN_STATS
    else None
)
if PRINT_STATS and STATISTICS is not None:
    print(
        f"Total burned area: {STATISTICS['total_burned_ha']:.3f} ha "
        f"[method={AREA_METHOD}, scale={ANALYSIS_SCALE} m, "
        f"CRS={STATISTICS['utm_crs']}]"
    )
    display(STATISTICS["severity_table"])
    if STATISTICS["admin_table"] is not None:
        print(
            f"Admin statistics preview (maximum "
            f"{ADMIN_STATS_PREVIEW_ROWS} rows):"
        )
        display(STATISTICS["admin_table"])


## 11. Google Drive export definitions

This cell mirrors the JavaScript raster/statistics exports but does not submit
anything unless `AUTO_START_EXPORT_TASKS = True` or you explicitly call
`start_export_tasks(...)`. If the optional statistics cell was skipped,
raster exports are still defined and statistics exports are omitted. In
the Earth Engine Python API, defining a batch task and starting it are
separate operations.


In [ ]:
def create_export_tasks(analysis, statistics):
    fire_date = analysis["fire_date"]
    tasks = {
        "burned": ee.batch.Export.image.toDrive(
            image=analysis["burned"].rename("burned"),
            description=f"S2_BurnedMask_dNBR2_{fire_date}_v19n",
            folder=EXPORT_FOLDER,
            fileNamePrefix=f"burned_{fire_date}_v19n",
            region=analysis["aoi"],
            scale=EXPORT_SCALE,
            crs=EXPORT_CRS,
            maxPixels=1e13,
        ),
        "severity": ee.batch.Export.image.toDrive(
            image=analysis["severity"].rename("severity"),
            description=f"S2_FireSeverity_{fire_date}_v19n",
            folder=EXPORT_FOLDER,
            fileNamePrefix=f"severity_{fire_date}_v19n",
            region=analysis["aoi"],
            scale=EXPORT_SCALE,
            crs=EXPORT_CRS,
            maxPixels=1e13,
        ),
    }

    if EXPORT_STATS and statistics is not None:
        tasks["severity_statistics"] = ee.batch.Export.table.toDrive(
            collection=statistics["severity_fc"],
            description=f"S2_BurnedArea_BySeverity_{fire_date}_v19n",
            folder=EXPORT_FOLDER,
            fileNamePrefix=f"burned_area_by_severity_{fire_date}_v19n",
            fileFormat="CSV",
        )
        if statistics["admin_fc"] is not None:
            tasks["admin_statistics"] = ee.batch.Export.table.toDrive(
                collection=statistics["admin_fc"],
                description=(
                    f"S2_BurnedArea_ByAdmin_L{ADMIN_LEVEL}_"
                    f"{fire_date}_v19n"
                ),
                folder=EXPORT_FOLDER,
                fileNamePrefix=(
                    f"burned_area_by_admin_L{ADMIN_LEVEL}_"
                    f"{fire_date}_v19n"
                ),
                fileFormat="CSV",
            )

    if RUN_MULTI_THRESH and MULTI_EXPORT:
        for threshold, threshold_outputs in analysis[
            "multi_threshold_outputs"
        ].items():
            tasks[f"threshold_{threshold}"] = (
                ee.batch.Export.image.toDrive(
                    image=threshold_outputs["final"],
                    description=(
                        f"S2_BurnedMask_dNBR2_{fire_date}_thr_{threshold}"
                    ),
                    folder=EXPORT_FOLDER,
                    fileNamePrefix=(
                        f"burned_{fire_date}_thr_{threshold}"
                    ),
                    region=analysis["aoi"],
                    scale=EXPORT_SCALE,
                    crs=EXPORT_CRS,
                    maxPixels=1e13,
                )
            )
    return tasks


def start_export_tasks(task_names=None):
    names = list(EXPORT_TASKS) if task_names is None else list(task_names)
    unknown = sorted(set(names) - set(EXPORT_TASKS))
    if unknown:
        raise KeyError(f"Unknown export task name(s): {unknown}")
    for name in names:
        EXPORT_TASKS[name].start()
        print(f"Started export: {name} (task id: {EXPORT_TASKS[name].id})")


STATISTICS_FOR_EXPORT = globals().get("STATISTICS")
if EXPORT_STATS and STATISTICS_FOR_EXPORT is None:
    print(
        "The optional statistics cell was skipped. Raster export definitions "
        "will be created; statistics export definitions will be omitted."
    )

EXPORT_TASKS = create_export_tasks(ANALYSIS, STATISTICS_FOR_EXPORT)
display(
    pd.DataFrame(
        [
            {
                "task_name": name,
                "description": task.config.get("description"),
                "submitted": False,
            }
            for name, task in EXPORT_TASKS.items()
        ]
    )
)

if AUTO_START_EXPORT_TASKS:
    start_export_tasks()
else:
    print(
        "Export tasks are defined but not submitted. To start selected tasks, "
        "run for example: start_export_tasks(['burned', 'severity'])"
    )


## 12. Checks, interpretation, and handoff notes

- The attached defaults are preserved: simple mode, 2026-07-29 target,
  2026-06-24 manual pre-date, full-date mosaics, SR imagery, cloud threshold
  60%, dNBR2 threshold 0.10, water/edge corrections, and the 25-pixel patch
  filter.
- A full-date mosaic includes every filtered Sentinel-2 granule intersecting
  the AOI on the selected UTC date. If exact dates are disabled, the closest
  available date within the configured search interval is selected first.
- Advanced median ranges keep Earth Engine's end-exclusive `filterDate`
  behavior from the JavaScript source.
- The source settings say `PATCH_FILTER_MODE = "pixel"` while one JavaScript
  helper tests the plural spelling `"pixels"`. This notebook consistently uses
  the documented singular value, so the default correctly applies
  `MIN_PATCH_PIXELS = 25`.
- SCL water is available only for Sentinel-2 SR/L2A. L1C runs fall back to the
  selected spectral water index when an SCL-based source is requested.
- Earth Engine web-tile URLs are temporary. Re-run the ArcGIS transfer cell if
  a transferred layer later stops drawing.
- ArcGIS Pro Scene layers are draped web-tile layers, not volumetric 3D data.
- The notebook does not start Drive exports by default. Starting an export is
  an explicit external action.
- Before committing notebook changes, restart the kernel, run cells in order,
  inspect the selected scenes and area totals, and clear all saved outputs.
- Validate burned-area and severity thresholds against known reference data
  before operational or decision-making use.


## 13. Download and validate burned area locally

This optional cross-check downloads the final burned mask as a **binary
GeoTIFF** (`0 = not burned`, `1 = burned`) in a metric projected CRS. It then:

1. calculates area locally from the count and area of burned raster pixels;
2. polygonizes and dissolves the burned pixels;
3. saves the burned polygon to a GeoPackage;
4. calculates polygon area locally; and
5. compares both local values with a fresh Earth Engine `pixelArea()` result.

Run this section after **Run the analysis**. It does not depend on the optional
statistics section or on ArcGIS Scene transfer. The output-folder field
below defaults to `Desktop\\wildfire_local_validation`; edit it before
running the download if you prefer another absolute path. A small difference can occur
at the AOI boundary when the download and reducer grids have different pixel
origins; both local methods should agree much more closely with each other.

Polygonization memory use grows with the number and complexity of burned
patches. If a very large or fragmented mask is slow, increase
`LOCAL_VALIDATION_SCALE` for a coarser diagnostic cross-check.


In [ ]:
# Run once only if the local-validation dependencies are missing.
# After installation, restart the kernel and rerun the notebook.
# %pip install geedim rasterio geopandas shapely pyproj


In [ ]:
# Edit this absolute path before running the download cell when desired.
local_validation_directory_input = widgets.Text(
    value=str(LOCAL_VALIDATION_DIRECTORY),
    description="Output folder:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="950px"),
)
reset_local_validation_directory_button = widgets.Button(
    description="Use Desktop default",
    icon="home",
)


def reset_local_validation_directory(_=None):
    local_validation_directory_input.value = str(
        Path.home() / "Desktop" / "wildfire_local_validation"
    )


reset_local_validation_directory_button.on_click(
    reset_local_validation_directory
)
display(
    widgets.HTML(
        "<b>Local validation output:</b> edit the absolute folder path "
        "below, then run the next cell."
    ),
    local_validation_directory_input,
    reset_local_validation_directory_button,
)


In [ ]:
def earth_engine_burned_area_hectares(analysis):
    """Recalculate the EE reference area without depending on STATISTICS."""
    area_image = ee.Image.pixelArea().rename("area")
    arguments = {
        "reducer": ee.Reducer.sum(),
        "geometry": analysis["aoi"],
        "scale": ANALYSIS_SCALE,
        "maxPixels": 1e13,
    }
    if AREA_METHOD == "utm":
        arguments["crs"] = utm_crs_for_geometry(analysis["aoi"])
    if TILE_SCALE is not None:
        arguments["tileScale"] = TILE_SCALE

    result = area_image.updateMask(analysis["burned"]).reduceRegion(
        **arguments
    )
    area_square_metres = ee.Number(
        ee.Algorithms.If(result.get("area"), result.get("area"), 0)
    )
    return float(area_square_metres.divide(10000).getInfo())


def download_and_validate_burned_area_locally(analysis):
    try:
        __import__("geedim")  # Required by geemap.download_ee_image.
        import geopandas as gpd
        import numpy as np
        import rasterio
        from pyproj import CRS
        from rasterio.features import shapes
        from shapely.geometry import shape
        from shapely.ops import unary_union
    except ImportError as error:
        raise RuntimeError(
            "Local validation dependencies are missing. Run the commented "
            "%pip install cell above, restart the kernel, and rerun the "
            "notebook."
        ) from error

    if LOCAL_VALIDATION_CONNECTIVITY not in (4, 8):
        raise ValueError("LOCAL_VALIDATION_CONNECTIVITY must be 4 or 8.")
    if LOCAL_VALIDATION_SCALE <= 0:
        raise ValueError("LOCAL_VALIDATION_SCALE must be positive.")

    configured_directory = (
        local_validation_directory_input.value.strip()
        if "local_validation_directory_input" in globals()
        else str(LOCAL_VALIDATION_DIRECTORY).strip()
    )
    if not configured_directory:
        raise ValueError("Choose a local validation output folder.")
    output_directory = Path(configured_directory).expanduser()
    if not output_directory.is_absolute():
        raise ValueError(
            "The local validation output folder must be an absolute path. "
            "Edit the field above or use the Desktop default."
        )
    output_directory = output_directory.resolve()
    try:
        output_directory.mkdir(parents=True, exist_ok=True)
    except PermissionError as error:
        raise PermissionError(
            f"Cannot write to {output_directory}. Edit the output-folder "
            "field above or use the Desktop default."
        ) from error
    fire_date = analysis["fire_date"]
    local_crs = (
        LOCAL_VALIDATION_CRS
        or utm_crs_for_geometry(analysis["aoi"]).getInfo()
    )
    crs_label = str(local_crs).replace(":", "")
    scale_label = f"{LOCAL_VALIDATION_SCALE:g}m"
    filename_stem = (
        f"burned_mask_{fire_date}_{crs_label}_{scale_label}"
    )
    raster_path = output_directory / f"{filename_stem}.tif"
    polygon_path = output_directory / f"{filename_stem}.gpkg"

    # Extend masked pixels as zero before clipping so the downloaded AOI is a
    # complete binary raster. Nearest-neighbour resampling preserves 0/1 values.
    binary_mask = (
        analysis["burned"]
        .unmask(value=0, sameFootprint=False)
        .clip(analysis["aoi"])
        .gt(0)
        .toUint8()
        .rename("burned")
    )

    if raster_path.exists() and not LOCAL_VALIDATION_OVERWRITE:
        print(f"Reusing existing raster: {raster_path}")
    else:
        print(
            f"Downloading binary mask at {LOCAL_VALIDATION_SCALE:g} m "
            f"in {local_crs}..."
        )
        geemap.download_ee_image(
            image=binary_mask,
            filename=str(raster_path),
            region=analysis["aoi"],
            crs=local_crs,
            scale=LOCAL_VALIDATION_SCALE,
            resampling="near",
            dtype="uint8",
            overwrite=True,
            num_threads=LOCAL_VALIDATION_NUM_THREADS,
            unmask_value=0,
        )

    if not raster_path.exists():
        raise RuntimeError(
            f"The binary mask download did not create {raster_path}."
        )

    with rasterio.open(raster_path) as source:
        raster_values = source.read(1)
        burned_pixels = raster_values == 1
        burned_pixel_count = int(np.count_nonzero(burned_pixels))
        raster_crs = source.crs
        raster_transform = source.transform

        if raster_crs is None:
            raise RuntimeError("The downloaded raster has no CRS.")
        projected_crs = CRS.from_user_input(raster_crs)
        if not projected_crs.is_projected:
            raise RuntimeError(
                "Local area validation requires a projected CRS. Leave "
                "LOCAL_VALIDATION_CRS=None to use the AOI-derived UTM CRS."
            )
        axis = projected_crs.axis_info[0]
        metres_per_unit = float(axis.unit_conversion_factor or 1)
        pixel_area_units = abs(
            raster_transform.a * raster_transform.e
            - raster_transform.b * raster_transform.d
        )
        pixel_area_square_metres = (
            pixel_area_units * metres_per_unit * metres_per_unit
        )
        raster_area_hectares = (
            burned_pixel_count * pixel_area_square_metres / 10000
        )

        burned_geometries = [
            shape(geometry)
            for geometry, value in shapes(
                burned_pixels.astype("uint8"),
                mask=burned_pixels,
                transform=raster_transform,
                connectivity=LOCAL_VALIDATION_CONNECTIVITY,
            )
            if int(value) == 1
        ]

    if not burned_geometries:
        raise RuntimeError(
            "The downloaded mask contains no burned pixels to polygonize."
        )

    dissolved_geometry = unary_union(burned_geometries)
    polygon_area_hectares = (
        float(dissolved_geometry.area)
        * metres_per_unit
        * metres_per_unit
        / 10000
    )

    if polygon_path.exists():
        if LOCAL_VALIDATION_OVERWRITE:
            polygon_path.unlink()
        else:
            raise FileExistsError(
                f"{polygon_path} exists. Set "
                "LOCAL_VALIDATION_OVERWRITE=True or change the directory."
            )
    polygon_frame = gpd.GeoDataFrame(
        {
            "burned": [1],
            "area_ha": [polygon_area_hectares],
            "source": ["Wildfire_detection_geemap"],
            "fire_date": [fire_date],
        },
        geometry=[dissolved_geometry],
        crs=raster_crs,
    )
    polygon_frame.to_file(
        polygon_path,
        layer="burned_area",
        driver="GPKG",
        index=False,
    )

    earth_engine_area_hectares = earth_engine_burned_area_hectares(
        analysis
    )
    comparison = pd.DataFrame(
        [
            {
                "method": "Earth Engine pixelArea reducer",
                "area_ha": earth_engine_area_hectares,
            },
            {
                "method": "Local binary-raster pixel count",
                "area_ha": raster_area_hectares,
            },
            {
                "method": "Local dissolved polygon",
                "area_ha": polygon_area_hectares,
            },
        ]
    )
    comparison["difference_from_ee_ha"] = (
        comparison["area_ha"] - earth_engine_area_hectares
    )
    if earth_engine_area_hectares:
        comparison["difference_from_ee_percent"] = (
            comparison["difference_from_ee_ha"]
            / earth_engine_area_hectares
            * 100
        )
    else:
        comparison["difference_from_ee_percent"] = float("nan")

    local_difference_percent = abs(
        raster_area_hectares - earth_engine_area_hectares
    ) / max(earth_engine_area_hectares, 1e-12) * 100
    tolerance_status = (
        "PASS"
        if local_difference_percent <= LOCAL_VALIDATION_TOLERANCE_PERCENT
        else "REVIEW"
    )

    print(f"Binary mask: {raster_path}")
    print(f"Burned polygon: {polygon_path}")
    print(
        f"Burned pixels: {burned_pixel_count:,}; "
        f"pixel area: {pixel_area_square_metres:.6f} m²"
    )
    print(
        f"Raster-vs-EE tolerance check: {tolerance_status} "
        f"({local_difference_percent:.6f}% difference; "
        f"limit {LOCAL_VALIDATION_TOLERANCE_PERCENT}%)"
    )
    display(comparison.style.format(
        {
            "area_ha": "{:,.6f}",
            "difference_from_ee_ha": "{:+,.6f}",
            "difference_from_ee_percent": "{:+.6f}%",
        }
    ))

    return {
        "binary_mask_path": raster_path,
        "polygon_path": polygon_path,
        "comparison": comparison,
        "burned_pixel_count": burned_pixel_count,
        "pixel_area_square_metres": pixel_area_square_metres,
        "tolerance_status": tolerance_status,
    }


LOCAL_AREA_VALIDATION = download_and_validate_burned_area_locally(ANALYSIS)
